# 06 · Разведка: перенос методов на новые типы требований

**Вопрос (H4: условное vs глобальное).** Steering, гейт, TACT и LoRA выбраны на val в 04 и учились на
требованиях «срок / количество / бюджет / наличие». Переносится ли их эффект на требования, которых не
было в train, — **рейтинг** и **гарантия**? Для контроля вторая половина набора — знакомые типы на свежем
seed, в том же формате ответа (все поля, включая `rating` и `warranty_months`).

**Как читать.** Разница прироста метода между половинами seen и unseen — его переносимость (один из
восьми критериев 2604.14090). Методы заморожены на val, на этом наборе ничего не подбирается; test-сплит
не используется. Результат **разведочный**.

> Всё до `fit` берётся из кэша 04 (те же конфиг вычислений, данные и модель).

In [ ]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz

In [ ]:
cfg = ExperimentConfig.make(
    name="transfer-qwen3-1.7b", hypothesis="explore", notebook="notebooks/06_transfer.ipynb",
    notes="Перенос выбранных на val методов на новые типы требований (рейтинг, гарантия).",
    overrides={
        "model.name": "Qwen/Qwen3-1.7B",
        "data.n_tasks": 160,
        "data.generator": "v3",
        "eval.methods": ["baseline", "strong_prompt", "restate", "explicit_state",
                         "steering", "steering_gated", "tact", "lora"],
        "steer.alphas": [0.25, 0.5, 1.0, 2.0, 4.0],
        "transfer.n_tasks": 100, "transfer.seed": 101,
    },
)
exp = Experiment(cfg)
exp.run("fit")
exp.results_table("val")

### Перенос: знакомые типы требований (seen) и новые (unseen)

In [ ]:
exp.transfer()
for g in ("seen", "unseen"):
    display(Markdown(f"**{g}**"))
    display(exp.results_table(f"transfer:{g}"))

In [ ]:
rows = []
for g in ("seen", "unseen"):
    for name, c in exp.state["transfer"][g]["comparisons"].items():
        rows.append({"group": g, "comparison": name, "diff": c["mean_diff"],
                     "ci": f"[{c['ci_low']:+.3f}, {c['ci_high']:+.3f}]", "excludes_zero": c["excludes_zero"]})
pd.DataFrame(rows)

In [ ]:
recs = {}
for g in ("seen", "unseen"):
    tasks = {t.task_id: t for t in exp.state["splits"][f"transfer:{g}"]}
    for m in ("baseline", "steering", "lora"):
        f = exp.dir / "decisions" / f"transfer-{g}__{m}.jsonl"
        df = pd.read_json(f, lines=True)
        df["need"] = [tasks[i].notes.split("need=")[1].split()[0] for i in df.task_id]
        recs[(g, m)] = df
pd.concat({k: v.groupby(["pair_type", "need", "branch"])["ok"].mean() for k, v in recs.items()}, axis=1).round(2)

## Вывод → журнал

In [ ]:
display(Markdown(exp.report(
    conclusion="""…""",
    next_step="""…""",
)))